In [ ]:
#install dependency
! pip install -q groq ipywidgets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.8/143.8 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 58.0 MB/s eta 0:00:00


In [ ]:

#API KEY INTEGRATION

import os
from getpass import getpass

if not os.environ.get("GROQ_API_KEY"):
  os.environ["GROQ_API_KEY"] = getpass("Enter your GROQ API Key: ")

print("API KEY SET." if os.environ.get("GROQ_API_KEY") else "API KEY NOT SET.")

Enter your GROQ API Key: ··········
API KEY SET.


In [ ]:
from groq import Groq
client = Groq()
for m in client.models.list().data:
  print(m.id)

meta-llama/llama-prompt-guard-2-22m
openai/gpt-oss-120b
meta-llama/llama-prompt-guard-2-86m
whisper-large-v3-turbo
openai/gpt-oss-safeguard-20b
canopylabs/orpheus-arabic-saudi
openai/gpt-oss-20b
allam-2-7b
canopylabs/orpheus-v1-english
qwen/qwen3.8-27b
whisper-large-v3


In [ ]:
#backend

from groq import Groq
from dataclasses import dataclass, field
from typing import List,Dict

@dataclass #USED FOR INPUT VALIDATION, str is enabled using this DECORATER

class Chatbot:
  model: str="openai/gpt-oss-20b"
  system_prompt: str= "You are an AI engineer."
  temperature: float = 0.7 #controls output , less then more imaginable and creative outputs,used to regulate behaviour
  max_tokens: int= 10240

  _client : Groq = field(default=None,repr=False)
  history: List[Dict[str,str]]= field(default_factory=list, repr=False)

  def __post_init__(self):
    self.client = Groq() #read api key from env

  def send(self, user_message: str) -> str:
    self.history.append({"role": "user", "content": user_message})
    messages=[{"role":"user", "content":self.system_prompt},*self.history]
    try:
      response = self.client.chat.completions.create(
          model=self.model,
          messages=messages,
          temperature=self.temperature,
          max_tokens=self.max_tokens)
    except Exception as e:
      self.history.pop()
      raise RuntimeError(f"Groq API error: {e}") from e

    #reply to user
    reply = response.choices[0].message.content
    self.history.append({"role":"assistant", "content":reply})
    return reply


ModuleNotFoundError: No module named 'groq'

In [ ]:
bot = Chatbot()
bot_msg=bot.send("create a code in java to check whether string is rotated version of other string")
print(bot_msg)

**Solution Overview**

A string `t` is a rotated version of a string `s` if we can take some prefix of `s` and append it to the end of the suffix of `s` so that the result equals `t`.  
A classic way to test this is:

1. The two strings must have the same length.  
2. If we concatenate `s` with itself (`s + s`), every possible rotation of `s` will appear as a contiguous substring.  
3. Therefore `t` is a rotation of `s` **iff** `t` is a substring of `s + s`.

The Java `String` class already provides `contains` (or `indexOf`) which runs in linear time, so the whole check is `O(n)` where `n` is the length of the strings.

---

### Java Implementation

```java
public class RotationChecker {

    /**
     * Returns true if t is a rotated version of s.
     *
     * @param s the original string
     * @param t the candidate rotated string
     * @return true if t is a rotation of s, false otherwise
     */
    public static boolean isRotation(String s, String t) {
        // Quick checks
  

In [ ]:
import ipywidgets as widgets
from IPython.display import display, HTML
import html
import traceback

bot = Chatbot()   # default

model_dropdown = widgets.Dropdown(
    options=["openai/gpt-oss-20b", "openai/gpt-oss-120b", "qwen/qwen3.8-27b", "allam-2-7b"],
    value=bot.model,
    desdescriptioncription="Model:",
    layout=widgets.Layout(width="300px"),
)

temp_slider = widgets.FloatSlider(
    value=bot.temperature, min=0.0, max=1.0, step=0.1,
    description="Temp:", continuous_update=False,
    layout=widgets.Layout(width="300px"),
)

chat_log = widgets.Output(
    layout=widgets.Layout(border="1px solid #ccc", height="400px", overflow_y="auto", padding="8px")
)
text_input = widgets.Text(placeholder="Type a message and press Enter...", layout=widgets.Layout(width="80%"))
send_button = widgets.Button(description="Send", button_style="primary")
clear_button = widgets.Button(description="Clear chat", button_style="warning")
status_label = widgets.Label(value="")

input_row = widgets.HBox([text_input, send_button, clear_button])
controls_row = widgets.HBox([model_dropdown, temp_slider])
ui = widgets.VBox([controls_row, chat_log, input_row, status_label])


def render_bubble(role, text):
    if role == "user":
        bg, color, align = "#DCF8C6", "#000000", "right"
    elif role == "assistant":
        bg, color, align = "#F1F0F0", "#000000", "left"
    else:  # error
        bg, color, align = "#FFD6D6", "#7A0000", "left"

    safe_text = html.escape(text).replace("\n", "<br>")
    bubble = f"""
    <div style="text-align:{align}; margin:6px 0;">
      <span style="display:inline-block; background:{bg}; color:{color}; padding:8px 12px;
                    border-radius:10px; max-width:75%; text-align:left;">
        <b>{role}:</b><br>{safe_text}
      </span>
    </div>
    """
    with chat_log:
        display(HTML(bubble))


def on_send(_=None):
    message = text_input.value.strip()
    if not message:
        return
    text_input.value = ""
    text_input.disabled = send_button.disabled = True
    status_label.value = "Waiting for response..."
    render_bubble("user", message)

    bot.model = model_dropdown.value
    bot.temperature = temp_slider.value

    try:
        reply = bot.send(message)
        render_bubble("assistant", reply)
        status_label.value = ""
    except Exception as e:
        render_bubble("error", str(e))
        status_label.value = "Error — see message above."
        traceback.print_exc()
    finally:
        text_input.disabled = send_button.disabled = False


def on_clear(_=None):
    bot.reset()
    chat_log.clear_output()
    status_label.value = "Chat cleared."


send_button.on_click(on_send)
clear_button.on_click(on_clear)
text_input.on_submit(on_send)

display(ui)

NameError: name 'Chatbot' is not defined